# Chapter 4 Test: Unicode Text Versus Bytes

Answer each question in the code cell(s) below it. Some ask for code, some for a written explanation, some for predicting output. **For prediction questions, reason through them before running.** The aim is not just to recall the API but to connect *why* text and bytes behave the way they do.

---

## Q1: Character, Code Point, Byte

Three terms are easy to blur together: **character**, **code point**, and **byte**.

a) Define each in one sentence, and state which of the three a Python `str` is a sequence of, and which a Python `bytes` object is a sequence of.

b) **Predict before running.** For the string `s = 'Zür​ich'` (that is a plain `ü` and a normal city name), what will `len(s)` and `len(s.encode('utf-8'))` be? Explain the source of any difference.

c) In one sentence: why is it wrong to say "a string is stored as its bytes"?

In [ ]:
# Your answer here
s = 'Zürich'


## Q2: Encode / Decode Round-Trips and Width

Take `word = 'déjà'`.

a) **Predict the byte length** of `word.encode(codec)` for each of `'utf-8'`, `'utf-16'`, and `'latin-1'`. Write your predicted numbers *before* running, then check.

b) One of those codecs is variable-width and one is fixed-width (per code unit). Say which is which, and explain why `'utf-8'` gives a different length than the character count while `'latin-1'` happens to match it here.

c) Encode `word` with `'latin-1'`, then decode those same bytes with `'utf-8'`. What happens, and what does that teach you about the relationship between bytes and the codec you *choose* to interpret them with?

In [ ]:
# Your answer here
word = 'déjà'


## Q3: `bytes`, `bytearray`, and the "item vs one-item slice" rule

Given:

```python
blob = bytes([104, 105, 33])   # three bytes
```

a) **Predict, without running:** what are the *types* and *values* of `blob[0]`, `blob[0:1]`, and `blob[-1:]`?

b) You saw in Chapter 2 that unpacking and slicing a container sequence deal in *references* to items. Here the items are plain `int`s in `0..255`. Explain in your own words why indexing (`blob[0]`) yields an `int` but slicing (`blob[0:1]`) yields another `bytes`. What general rule about *slices of a built-in sequence* connects this to `list` and `str`?

c) `bytes` is immutable; `bytearray` is mutable. Convert `blob` to a `bytearray`, change its first byte to the value of ASCII `'H'` (72), and show it now spells something different — without creating a new object (verify with `id`).

In [ ]:
# Your answer here
blob = bytes([104, 105, 33])


## Q4: Handling Encode / Decode Errors

Two failing operations:

```python
# (1) encoding side
'São Paulo café ☕'.encode('cp437')

# (2) decoding side
b'\xff\xfe\x00 caf\xe9'.decode('utf-8')
```

a) Which exception does each raise, and *why* — i.e. what does the codec fail to do in each direction?

b) Make **(1)** succeed three different ways, using `errors='ignore'`, `errors='replace'`, and `errors='xmlcharrefreplace'`. Show all three outputs and describe what each strategy sacrifices or preserves.

c) Make **(2)** succeed with `errors='replace'`, then answer: replacing on *decode* can silently corrupt data. Which side of the "Unicode sandwich" (decode-on-input vs encode-on-output) is more dangerous to be lenient on, and why?

In [ ]:
# Your answer here


## Q5: The BOM and the UTF-16 Variants

Encode `text = 'hi'` with `'utf-16'`, `'utf-16-le'`, and `'utf-16-be'`.

a) **Predict** the exact bytes for each of the three, then check. What extra two bytes appear in the plain `'utf-16'` output that are absent from the `-le` / `-be` variants, and what is that marker called?

b) Explain what the BOM communicates and how a reader uses it to decide between little-endian and big-endian ordering. What does specifying `-le` or `-be` explicitly let you skip?

c) Why does UTF-8 normally carry no BOM at all? (Connect your answer to the fact that UTF-8 is defined over single bytes, not multi-byte code units.)

In [ ]:
# Your answer here
text = 'hi'


## Q6: Handling Text Files — the Unicode Sandwich

Complete the exercise below.

a) Write the string `msg = 'costs 5€ — 便宜'` to a file **without** passing `encoding=`, then read it back **without** passing `encoding=`. On some machines this round-trips fine; on others it corrupts or raises. Print `locale.getpreferredencoding()` and explain why relying on the default makes this code non-portable.

b) Now do it *correctly*: write and read the same string using `encoding='utf-8'` explicitly on both ends, and confirm the round-trip is exact.

c) State the "Unicode sandwich" rule in one sentence, and identify exactly *where* the decode and the encode happen in your part (b) code. Why should the `str` layer in the middle never have to think about bytes?

In [ ]:
# Your answer here
import locale
from pathlib import Path

msg = 'costs 5€ — 便宜'
path = Path('_ch04_scratch.txt')


## Q7: Normalization and Equality

Build two strings that *look* identical but are composed differently:

```python
a = 'e\u0301'   # 'e' followed by COMBINING ACUTE ACCENT
b = '\u00e9'    # single precomposed 'é'
```

a) **Predict:** does `a == b`? What are `len(a)` and `len(b)`? Explain why two strings that render the same on screen can be unequal — connect this to your Q1 answer that a `str` is a sequence of *code points*, not glyphs.

b) Use `unicodedata.normalize` to make them compare equal. Show that `NFC` collapses `a` toward the precomposed form and `NFD` expands `b` toward the combining form. Which normal form would you store in a database key, and why?

c) In one sentence: why is normalization a *prerequisite* for meaningful equality and de-duplication of user-entered text?

In [ ]:
# Your answer here
import unicodedata

a = 'e\u0301'
b = '\u00e9'


## Q8: Case Folding vs Lowercasing

a) Write a function `same_text(x, y)` that reports whether two strings are equal *ignoring case*, robustly enough to handle non-English text. Use `str.casefold()`, and also normalize (NFC) so combining-vs-precomposed forms don't defeat you — connecting this back to Q7.

b) Demonstrate a case where `str.casefold()` and `str.lower()` disagree. A classic example is the German `'ß'` (`'\u00df'`): show what `.lower()` and `.casefold()` each produce for `'STRAßE'` vs `'strasse'`, and explain which method is correct for *matching* and which merely changes display case.

c) Test `same_text` on `('Café', 'cafe\u0301')` and on `('STRAßE', 'strasse')` and confirm both return `True`.

In [ ]:
# Your answer here
import unicodedata

def same_text(x, y):
    ...


## Q9: Sorting Unicode Text

Given a list of Portuguese fruit names with accents:

```python
fruits = ['maçã', 'açaí', 'abacaxi', 'ável', 'amora']
```

a) Sort it with plain `sorted(fruits)`. Where does an accented word like `'açaí'` or `'ável'` land relative to the plain-ASCII words, and why? (Connect this to the fact that the default comparison walks *code points*, and accented letters sit far above `a`–`z` in the code space.)

b) Produce a linguistically correct ordering. Do it with `locale.strxfrm` as a sort key *after* setting an appropriate locale — and note in a comment that this mutates global state and may fail if the locale isn't installed. Then briefly describe how the `pyuca` library (Unicode Collation Algorithm) solves the same problem *without* touching the OS locale.

c) In one sentence: why is "correct" string sorting a *linguistic* question rather than a purely numeric one?

In [ ]:
# Your answer here
import locale

fruits = ['maçã', 'açaí', 'abacaxi', 'ável', 'amora']


## Q10: `memoryview` — Editing Bytes Without Copying

This ties the whole chapter back to the reference-vs-value idea from Chapter 2.

```python
frame = bytearray(b'\x00\x01\x02\x03\x04\x05')
```

a) Create a `memoryview` over `frame`. Take a slice of the view covering the middle two bytes and assign new byte values *through the view*. **Predict first:** does the original `frame` change? Show `id(frame)` before and after to confirm no new object was created.

b) Contrast this with `middle = frame[2:4]` (ordinary bytearray slicing). Modify `middle` and show that `frame` is **unaffected**. Explain the difference: which operation *copies* the bytes and which one shares the *same underlying buffer*?

c) Connect the dots: in Chapter 2 you learned that unpacking/slicing a container copies *references*, not the objects themselves. A `memoryview` goes one step further — it shares the *raw buffer* itself. In one or two sentences, explain why that makes `memoryview` valuable when processing large binary data (images, network frames), and what the danger is (a mutation through one view is visible everywhere).

In [ ]:
# Your answer here
frame = bytearray(b'\x00\x01\x02\x03\x04\x05')


---

# Hands-On Project: A Mini "Guestbook Ingestion" Pipeline

The questions above isolated each concept. This project makes you **combine them** the way a real codebase forces you to. You will build a small pipeline that ingests guestbook entries arriving as **raw bytes from mixed, messy sources** — different encodings, some corrupt, some using combining accents, mixed case — and turns them into a clean, sorted, human-readable report written back to disk.

This is deliberately close to real life: **input is bytes you don't control, the middle is `str` you reason about, output is bytes you choose the encoding for.** That is the Unicode sandwich as an actual program, not a slogan.

## The scenario

A guestbook web form in three different countries posts entries to you. Because the front-ends are old and inconsistent, the raw payloads reach you as `bytes` in **different encodings**, and one client is buggy and sends garbage. Each decoded line looks like:

```
name|country|message
```

Your job, end to end:

1. **Decode** each raw payload with the encoding its source declares — robustly, so one corrupt client can't crash the whole batch.
2. **Parse** each decoded blob into `(name, country, message)` records (multiple lines per payload possible).
3. **Normalize** names so that `'José'` typed with a combining accent and `'José'` typed precomposed are treated as the *same person*, and de-duplicate case-insensitively (`'jose'` vs `'José'` is a design decision you'll make and justify).
4. **Sort** the unique visitors for a report using a linguistically-aware key (not raw code points).
5. **Emit** a UTF-8 report file, and also a compact binary "index" using `bytearray` + `memoryview` that records each name's byte length without copying the buffer.

Work through the numbered cells. Each cell states its acceptance check. Fill in the `...` / `TODO`s. Don't peek at a reference solution — struggling here is the point.

Run the setup cell first.

## Setup — the raw inbound payloads (given)

Run this cell as-is. It fabricates the messy input you have to deal with. Note that these are `bytes`, produced with **different encodings on purpose**, plus one intentionally-corrupt payload. You did not choose these encodings — the clients did.

In [ ]:
# === Setup: run as-is. These simulate raw uploads from three flaky clients. ===
import unicodedata

# A payload is (declared_encoding, raw_bytes). The declared encoding is what the
# client CLAIMS it used — you must trust it to decode, but be ready for it to lie.

# Client A (Brazil): UTF-8, two entries. Note 'José' here uses a COMBINING accent:
#   'e' + U+0301, so it will NOT be == a precomposed 'José' until you normalize.
client_a = (
    'utf-8',
    ('Jose\u0301|BR|Adorei o cafe\u0301!\n'
     'Ana|BR|Muito obrigada 😊').encode('utf-8'),
)

# Client B (Germany): latin-1 (a.k.a. ISO-8859-1), one entry with an umlaut.
client_b = (
    'latin-1',
    'Jörg|DE|Schöne Grüße'.encode('latin-1'),
)

# Client C (France): UTF-16, one entry — precomposed 'José' this time (U+00E9),
#   plus a duplicate of Ana in a different case to test your de-duplication.
client_c = (
    'utf-16',
    ('José|FR|Merci beaucoup\n'
     'ana|FR|encore une fois').encode('utf-16'),
)

# Client D: BUGGY. It declares utf-8 but sends bytes that are not valid utf-8.
client_d = (
    'utf-8',
    b'Zo\xeb|NL|corrupt \xff\xfe payload',   # \xeb, \xff, \xfe are illegal utf-8 here
)

RAW_PAYLOADS = [client_a, client_b, client_c, client_d]

print(f'{len(RAW_PAYLOADS)} raw payloads ready.')
for enc, raw in RAW_PAYLOADS:
    print(f'  declared={enc:<8} {len(raw):>3} bytes  {raw[:24]!r}...')

### Step 1 — Decode at the boundary (the top slice of the sandwich)

Write `decode_payload(declared_encoding, raw)` that turns raw `bytes` into a `str`.

Requirements:
- Decode using the **declared** encoding.
- If decoding fails (client D), **do not crash the batch**. Instead, decode with `errors='replace'` so you keep as much text as possible, and record that this payload was lossy. Return `(text, was_lossy)`.
- Do **not** use a bare `except:` that hides the fact that something went wrong — you want to *know* it was lossy.

Think about *why* you decode here and nowhere deeper: this is the only place bytes should turn into text. Everything downstream works on `str`.

**Acceptance:** clients A/B/C decode cleanly (`was_lossy == False`); client D decodes with replacement characters and `was_lossy == True`, without raising.

In [ ]:
def decode_payload(declared_encoding, raw):
    """Return (text, was_lossy). Never raise on bad bytes."""
    # TODO: implement
    ...


# --- self-check (leave as-is) ---
for enc, raw in RAW_PAYLOADS:
    text, lossy = decode_payload(enc, raw)
    tag = 'LOSSY' if lossy else 'ok'
    print(f'[{tag:>5}] {text!r}')

### Step 2 — Parse decoded text into records

Now that everything is `str`, write `parse_records(text)` that splits a decoded payload into a list of `(name, country, message)` tuples.

Requirements:
- One record per non-empty line.
- Split each line on `'|'` into exactly three fields; strip surrounding whitespace from each field.
- Skip (don't crash on) malformed lines that don't have exactly three fields.

Notice you are working purely in text here — no encoding concerns at all. That separation is the payoff of Step 1.

**Acceptance:** the four decoded payloads together yield the visitor rows (José, Ana, Jörg, José, ana, and the corrupted "Zoë"-ish row). The corrupt row still parses into *some* three fields even though its message contains replacement characters.

In [ ]:
def parse_records(text):
    """Return a list of (name, country, message) tuples from decoded text."""
    # TODO: implement
    ...


# --- build the full record set from all payloads (leave as-is) ---
records = []
for enc, raw in RAW_PAYLOADS:
    text, _lossy = decode_payload(enc, raw)
    records.extend(parse_records(text))

for r in records:
    print(r)
print(f'\n{len(records)} records parsed.')

### Step 3 — Normalize + de-duplicate visitors

You have `'Jose\u0301'` (combining) from client A and `'José'` (precomposed) from client C. Rendered, they're the same person — but as raw `str` they are **not equal** and have **different `len`**. You also have `'Ana'` and `'ana'`.

Write two helpers:

- `match_key(name)` — the key you use to decide *"is this the same visitor?"*. It must:
  - NFC-normalize so combining vs precomposed forms collapse (connects to Q7),
  - `casefold()` so case differences collapse for matching (connects to Q8).
- `display_name(name)` — the form you keep for showing to humans. Keep the visitor's original capitalization, but still NFC-normalize so the stored text is canonical (connects to Q7).

Then build `unique_visitors`: a dict mapping `match_key(name) -> display_name(name)`, keeping the **first** display form seen for each key.

**Design question to answer in a comment:** by folding case in the match key, you merge `'Ana'` and `'ana'` into one visitor. Is that the right call for a guestbook? State your assumption. (There's no single right answer — the point is that *normalization policy is a decision, not a default.*)

**Acceptance:** José (both forms) collapses to ONE visitor; Ana/ana collapses to ONE; you end up with fewer unique visitors than raw records.

In [ ]:
def match_key(name):
    """Key used to decide identity: NFC + casefold."""
    # TODO: implement
    ...


def display_name(name):
    """Human-facing form: keep original case, but NFC-normalize."""
    # TODO: implement
    ...


# Design decision (edit this comment with your reasoning):
# TODO: is case-folding names the right policy for a guestbook? Why / why not?

# --- build unique visitors, keeping first display form per key (leave as-is) ---
unique_visitors = {}
for name, country, message in records:
    key = match_key(name)
    if key not in unique_visitors:
        unique_visitors[key] = display_name(name)

print(f'{len(records)} raw records -> {len(unique_visitors)} unique visitors')
for key, disp in unique_visitors.items():
    print(f'  key={key!r:<12} display={disp!r}')

### Step 4 — Sort visitors linguistically

Produce `sorted_names`: the display names sorted for a human-readable report.

Requirements:
- First show what **naive** `sorted(...)` does (raw code-point order) so you can see accented names misordering — connects to Q9.
- Then produce a **locale-aware** ordering using `locale.strxfrm` as the sort key. Wrap the locale setup in `try/except` and fall back gracefully if the locale isn't installed on this machine (print a note). In the fallback, mention that `pyuca` would give correct collation *without* depending on OS locales.

**Acceptance:** you print both orderings and can articulate at least one pair whose order differs (or would differ) between naive and linguistic sorting.

In [ ]:
import locale

names = list(unique_visitors.values())

# 1) Naive, code-point ordering:
naive_sorted = sorted(names)
print('naive     :', naive_sorted)

# 2) Locale-aware ordering (fill in), with graceful fallback:
def locale_sorted(items):
    # TODO: set an appropriate locale (e.g. a UTF-8 locale), then
    #       return sorted(items, key=locale.strxfrm).
    #       On failure, print a note (mention pyuca) and return naive order.
    ...


sorted_names = locale_sorted(names)
print('linguistic:', sorted_names)

### Step 5 — Emit: a UTF-8 text report + a compact binary index

This is the bottom slice of the sandwich — text turns back into bytes on the way out.

**5a. Text report.** Write `write_report(path, names)` that writes one visitor name per line to `path`, encoded **explicitly** as UTF-8 (connects to Q6 — never rely on the platform default). Then read it back with the same explicit encoding to prove round-trip.

**5b. Binary index.** For each name, record its UTF-8 **byte length** in a compact binary buffer:
- Build a single `bytearray` `index` of one byte per name (assume each fits in a byte).
- Then create a `memoryview` over `index` and use it to bump the *last* entry by 1 **in place**, without copying the buffer (connects to Q10 — `memoryview` shares storage; a slice would copy). Confirm the change is visible through the original `bytearray`.

**Acceptance:** the file round-trips exactly (`read == written`), and mutating through the `memoryview` changes `index` itself (same object, no copy).

In [ ]:
from pathlib import Path

report_path = Path('_ch04_guestbook_report.txt')

# 5a. Explicit-UTF-8 text report + round-trip check
def write_report(path, names):
    """Write one name per line, explicitly UTF-8 encoded."""
    # TODO: implement (open with encoding='utf-8')
    ...


write_report(report_path, sorted_names)
read_back = report_path.read_text(encoding='utf-8')       # explicit on the way in too
written = '\n'.join(sorted_names) + '\n'                    # adjust to match your writer
print('round-trip ok:', read_back == written)
print('--- file contents ---')
print(read_back)

# 5b. Compact binary index of UTF-8 byte lengths, mutated via memoryview
index = bytearray(len(n.encode('utf-8')) for n in sorted_names)
print('index before:', list(index))

view = memoryview(index)
# TODO: bump the LAST entry by 1 through `view` (in place, no copy)
...

print('index after :', list(index))
print('same object mutated (no copy):', True)  # replace with a real check if you like

### Reflection — map each step back to the chapter

Before checking anything, answer in your own words:

1. **Step 1 (decode):** Why is the "Unicode sandwich" a discipline and not just a metaphor? What breaks if you let bytes leak past this boundary?
2. **Step 3 (normalize):** `'Jose\u0301'` vs `'José'` — which Unicode concept made them unequal, and which function reconciled them? Why did you *also* casefold for matching but *not* for display?
3. **Step 4 (sort):** Why does `sorted()` misorder accented names, and what does `locale.strxfrm` (or `pyuca`) change? What's the hidden cost of the `locale` approach?
4. **Step 5 (emit):** Why pass `encoding='utf-8'` explicitly on *both* write and read? And why did the `memoryview` mutation change `index` while a slice would not have — which earlier chapter's reference-vs-value rule is this?

There are no cells to fill in here — this is the "can I explain it?" check that turns the exercise into understanding.